# Locked source-study evaluation on Kaggle

This notebook runs the corrected, accession-labelled evaluation. Both PCBert-Kla and token-gated MM-Kla use the same Gao 2020 training, validation, and internal-test partitions, followed by the same held-out rice study. It does not reuse the overlap-affected original test set. The source-data rules and limitations are recorded in `review_data/locked_source_study_v1/README.md`.

Kaggle settings: select **GPU T4 x2** and enable **Internet**. Add a fine-grained, read-only GitHub token as the Kaggle secret `GITHUB_TOKEN`; grant this notebook access to that secret. The token is used only for cloning the private repository and is never placed in the repository URL, notebook output, or a dataset. No separate dataset upload is needed because the locked manifests are versioned with the code. The current runner uses one GPU per process; the two models run sequentially to keep the initial comparison simple and reproducible.

## 1. Obtain and record the working copy

The repository is cloned once into `/kaggle/working`. Re-running this cell keeps the existing checkout rather than silently changing its commit. The Git commit printed below identifies the code used for both runs.

In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'MM-Kla'
REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
PROJECT_DIR = REPO_DIR / 'experiment' / 'pcbert_kla_clean'

if not WORKING_DIR.is_dir():
    raise RuntimeError('This notebook requires a Kaggle notebook runtime.')

if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing or empty.')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" x-access-token ;;\n'
            '  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n'
            '  *) exit 1 ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({
            'GIT_ASKPASS': str(askpass),
            'GIT_TERMINAL_PROMPT': '0',
            'MM_KLA_GITHUB_TOKEN': token,
        })
        try:
            clone = subprocess.run(
                ['git', '-c', 'credential.helper=', 'clone', '--depth', '1',
                 '--branch', 'main', REPO_URL, str(REPO_DIR)],
                env=env, capture_output=True, text=True, check=False,
            )
        finally:
            env.pop('MM_KLA_GITHUB_TOKEN', None)
            del token
    if clone.returncode != 0:
        raise RuntimeError(
            'Private clone failed. Check Kaggle Internet, GITHUB_TOKEN access, '
            'and the repository name. The Git error was not printed to protect the token.'
        )
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout.')

if not (PROJECT_DIR / 'scripts' / 'run_replication.py').is_file():
    raise RuntimeError(f'Project runner not found under {PROJECT_DIR}.')
GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR,
    capture_output=True, text=True, check=True,
).stdout.strip()
print('Repository commit:', GIT_COMMIT)
print('Project directory:', PROJECT_DIR)

## 2. Install and record the runtime

The repository requirements provide the packages used by the source-study runner. The manifest below records the actual versions installed in this Kaggle session. Each T4 has its own memory; GPU T4 x2 does not provide one combined 32-GB device.

In [ ]:
import importlib.metadata
import json
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    cwd=PROJECT_DIR, check=True,
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available; enable a GPU in Kaggle settings.')
runtime = {
    'git_commit': GIT_COMMIT,
    'python': sys.version.split()[0],
    'cuda_device_count': torch.cuda.device_count(),
    'cuda_devices': [
        {'name': torch.cuda.get_device_name(index),
         'memory_gb': round(torch.cuda.get_device_properties(index).total_memory / 2**30, 2)}
        for index in range(torch.cuda.device_count())
    ],
    'packages': {name: importlib.metadata.version(name) for name in
                 ('biopython', 'numpy', 'pandas', 'scikit-learn',
                  'torch', 'transformers', 'sentencepiece')},
}
RUNTIME_PATH = WORKING_DIR / 'mm_kla_kaggle_runtime.json'
RUNTIME_PATH.write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2, sort_keys=True))

## 3. Validate the locked data

This check verifies the input-file hashes, accession and cluster assignments, exact-window separation, sample counts, and computed physicochemical features. It does not download ProtBert or train a model.

In [ ]:
import hashlib

LOCK_PATH = PROJECT_DIR / 'review_data/locked_source_study_v1/protocol_lock.json'
LOCK_SHA256 = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()
subprocess.run(
    [sys.executable, 'scripts/run_replication.py', '--run', 'source-study-check'],
    cwd=PROJECT_DIR, check=True,
)
print('Locked protocol SHA-256:', LOCK_SHA256)

## 4. Matched model runs

Both runs use ProtBert, four trainable encoder layers, SGD at learning rate 0.003, batch size 4, 30 epochs, seed 42, Gao-validation AUPRC for checkpoint selection, and a fixed decision threshold of 0.5. The only planned difference is the classifier architecture. Gao internal and rice external results are generated **after** checkpoint selection. The rice set must not be used to change these settings and then described as untouched external validation.

The default is to execute both models sequentially on one GPU. If time or quota requires separate sessions, change `ARCHITECTURES_TO_RUN` to one architecture and preserve each completed run's output directory before the session expires. These first runs do not satisfy the review request for multiple independent seeds; that is a later, predeclared extension of the same locked protocol.

In [ ]:
ARCHITECTURES_TO_RUN = ('baseline', 'token_gated')
SEED = 42
OUTPUT_ROOT = PROJECT_DIR / 'outputs' / 'kaggle_source_study'
EXPECTED_SETTINGS = {
    'run': 'source-study', 'model_name': 'Rostlab/prot_bert',
    'encoder_layers': 4, 'epochs': 30, 'batch_size': 4,
    'device': 'cuda:0', 'optimizer': 'sgd',
    'learning_rate': 0.003, 'weight_decay': 0.0,
    'scheduler': 'none', 'seed': SEED,
    'monitor_metric': 'AUPRC', 'decision_threshold': 0.5,
}

def matching_run_dirs(architecture):
    root = OUTPUT_ROOT / 'source_study' / 'runs'
    matches = []
    for config_path in root.glob('*/run_config.json'):
        config = json.loads(config_path.read_text())
        if (config.get('architecture') == architecture
                and all(config.get(key) == value for key, value in EXPECTED_SETTINGS.items())
                and config.get('source_lock_sha256') == LOCK_SHA256):
            matches.append(config_path.parent)
    return sorted(matches, key=lambda path: path.stat().st_mtime, reverse=True)

RUN_DIRS = {}
for architecture in ARCHITECTURES_TO_RUN:
    command = [
        sys.executable, 'scripts/run_replication.py',
        '--run', 'source-study', '--architecture', architecture,
        '--model-name', 'Rostlab/prot_bert',
        '--encoder-layers', '4', '--epochs', '30',
        '--batch-size', '4', '--device', 'cuda:0',
        '--optimizer', 'sgd', '--learning-rate', '0.003',
        '--weight-decay', '0', '--scheduler', 'none',
        '--seed', str(SEED), '--output-dir', str(OUTPUT_ROOT),
        '--save-models',
    ]
    print('Running:', architecture, 'on cuda:0', flush=True)
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    matches = matching_run_dirs(architecture)
    if not matches:
        raise RuntimeError(f'No saved run found for {architecture}.')
    RUN_DIRS[architecture] = matches[0]
    print('Saved run:', RUN_DIRS[architecture], flush=True)

## 5. Generated result table

This table reads the saved JSON metrics from the completed runs. No scores are entered by hand. The rice controls are unannotated lysines and remain putative negatives; rice results represent cross-study, cross-species transfer. Published PCBert-Kla numbers from the original overlapping benchmark are not directly comparable with these corrected-split scores.

In [ ]:
import pandas as pd
from IPython.display import display

metric_names = ('ACC', 'AUC', 'AUPRC', 'F1', 'MCC', 'Pre', 'Rec', 'SP')
rows = []
for architecture, run_dir in RUN_DIRS.items():
    config = json.loads((run_dir / 'run_config.json').read_text())
    for split_name, filename in (
        ('Gao internal test', 'test_metrics.json'),
        ('Rice study-held-out test', 'rice_external_metrics.json'),
    ):
        metrics = json.loads((run_dir / filename).read_text())
        rows.append({
            'architecture': architecture, 'split': split_name,
            'seed': config['model_seed'],
            'selected_epoch': config['selected_epoch'],
            **{name: metrics[name] for name in metric_names},
        })
if not rows:
    raise RuntimeError('No completed model runs are available for comparison.')
results = pd.DataFrame(rows)
display(results.round(4))
SUMMARY_PATH = WORKING_DIR / 'mm_kla_source_study_metrics.csv'
results.to_csv(SUMMARY_PATH, index=False)
print('Saved summary:', SUMMARY_PATH)
print('Runtime record:', RUNTIME_PATH)
print('Per-run predictions, metrics, and checkpoints:', OUTPUT_ROOT / 'source_study' / 'runs')